In [34]:
import pandas as pd
data = pd.read_csv(r'./data/America.csv', index_col=0, parse_dates=True, dtype=object)
data = data.loc[data["code"].isin(["AAPL","ADBE","BABA","SNE","V"])]
features = ["close","high","low","open"]
data[features] = data[features].astype(float)
data

,close,open,high,low,volume,percent,code
time,,,,,,,
2015-01-05,106.25,108.29,108.65,105.41,64.29M,-2.82%,AAPL
2015-01-06,106.26,106.54,107.43,104.63,65.80M,0.01%,AAPL
2015-01-07,107.75,107.20,108.20,106.69,40.11M,1.40%,AAPL
2015-01-08,111.89,109.23,112.15,108.70,59.36M,3.84%,AAPL
2015-01-09,112.01,112.67,113.25,110.21,53.70M,0.11%,AAPL
...,...,...,...,...,...,...,...
2017-12-22,45.68,45.73,45.78,45.44,453.37K,0.59%,SNE
2017-12-26,45.21,45.34,45.46,45.13,584.26K,-1.03%,SNE
2017-12-27,45.26,45.20,45.27,45.07,386.53K,0.11%,SNE


In [35]:
start_date = "2015-01-05"
end_date = "2016-12-31"
mask = (data.index >= start_date) & (data.index <= end_date)
data = data[mask]
data

,close,open,high,low,volume,percent,code
time,,,,,,,
2015-01-05,106.25,108.29,108.65,105.41,64.29M,-2.82%,AAPL
2015-01-06,106.26,106.54,107.43,104.63,65.80M,0.01%,AAPL
2015-01-07,107.75,107.20,108.20,106.69,40.11M,1.40%,AAPL
2015-01-08,111.89,109.23,112.15,108.70,59.36M,3.84%,AAPL
2015-01-09,112.01,112.67,113.25,110.21,53.70M,0.11%,AAPL
...,...,...,...,...,...,...,...
2016-12-23,28.46,28.36,28.49,28.36,331.43K,0.39%,SNE
2016-12-27,28.38,28.30,28.48,28.29,402.47K,-0.28%,SNE
2016-12-28,28.28,28.50,28.50,28.27,355.88K,-0.35%,SNE


In [36]:
asset_dict = dict()
date_len = len(data.index.unique())
date_len # 日期长度

504

In [37]:
pd.set_option('future.no_silent_downcasting', True) # 设置Pandas选项来禁用静默向下转换，解决填充警告
for asset in ["AAPL","ADBE","BABA","SNE","V"]:
    asset_data = data[data["code"] == asset].reindex(data.index.unique()).sort_index()
    # asset_data['close'] = asset_data['close'].fillna(method='pad')
    
    # 预处理（填充缺失数据、删除code列）
    asset_data = asset_data.bfill(axis=1)  # 后向填充
    asset_data = asset_data.ffill(axis=1)  # 前向填充
    
    # 归一化（最后一天收盘价为基准）
    base_price = asset_data['close'].iloc[-1]
    asset_data['close'] = asset_data['close'] / base_price
    asset_data['high'] = asset_data['high'] / base_price
    asset_data['low'] = asset_data['low'] / base_price
    asset_data['open'] = asset_data['open'] / base_price
    # 以字典形式保存处理好的数据
    asset_dict[str(asset)] = asset_data
    
asset_dict

{'AAPL':                close      open      high       low  volume percent  code
 time                                                                    
 2015-01-05  0.917372  0.934985  0.938094  0.910119  64.29M  -2.82%  AAPL
 2015-01-06  0.917458  0.919876   0.92756  0.903385  65.80M   0.01%  AAPL
 2015-01-07  0.930323  0.925574  0.934208  0.921171  40.11M   1.40%  AAPL
 2015-01-08  0.966068  0.943101  0.968313  0.938525  59.36M   3.84%  AAPL
 2015-01-09  0.967104  0.972803   0.97781  0.951563  53.70M   0.11%  AAPL
 ...              ...       ...       ...       ...     ...     ...   ...
 2016-12-23  1.006044  0.998014  1.006044  0.998014  14.25M   0.20%  AAPL
 2016-12-27  1.012433  1.006044  1.017095  1.005785  18.30M   0.64%  AAPL
 2016-12-28  1.008116  1.014678  1.018995  1.003281  20.91M  -0.43%  AAPL
 2016-12-29  1.007857  1.005439  1.011138  1.005008  15.04M  -0.03%  AAPL
 2016-12-30       1.0  1.007166  1.011915  0.996633  30.59M  -0.78%  AAPL
 
 [504 rows x 7 columns],
 'A

In [64]:
import numpy as np
states = []  # 存储所有状态张量的列表
y0=np.array([[1],[1],[1],[1],[1],[1]])
price_history = [y0]  # 存储价格历史用于计算回报的列表
t = 1 # 时刻指针
M = 6 # 股票数
L = 1 # 时间窗口长度
N = 4 # CHLO特征数

In [65]:
while t <= date_len: # 【1~504】
    # 初始化各个特征矩阵，第一行是现金资产（值始终为1）
    V_close = np.ones(L)  # 收盘价特征矩阵，现金资产始终为1
    V_high = np.ones(L)   # 最高价矩阵
    V_open = np.ones(L)  # 开盘价矩阵
    V_low = np.ones(L)  # 最低价矩阵
    y = np.ones(1)  # 收盘价涨幅变化
    
    for asset in  ["AAPL","ADBE","BABA","SNE","V"]:
        asset_data = asset_dict[str(asset)]  # 获取该资产的数据

        # 堆叠各个特征的历史数据
        # np.vstack() - 垂直堆叠数组
        # asset_data.iloc[t - self.L - 1:t - 1] - 获取从t-L-1到t-1的历史数据【窗口为1也可以直接取值】
        V_close = np.vstack((V_close, asset_data.iloc[t - L :t ]['close'].values))
        V_high = np.vstack((V_high, asset_data.iloc[t - L :t ]['high'].values))
        V_low = np.vstack((V_low, asset_data.iloc[t - L :t ]['low'].values))
        V_open = np.vstack((V_open, asset_data.iloc[t - L :t ]['open'].values))
        if t != 1:
            y = np.vstack((y, asset_data.iloc[t-1]['close'] / asset_data.iloc[t-2]['close']))
    state = V_close
    # np.stack() - 沿着新轴堆叠数组，axis=2表示在第三个维度堆叠【特征】
    state = np.stack((state, V_high, V_low, V_open), axis=2)
    state = state.reshape(1, M, L, N)
    states.append(state)  # 保存状态到列表
    if t != 1:
        price_history.append(y)  # 保存价格变化到列表
    t = t + 1  # 移动到下一个时间点
# 重置环境，开始训练
t=1
START_FLAG = True

In [66]:
# state
# state = [
#     # 资产0：现金
#     [[1.0, 1.0, 1.0, 1.0]],  # [收盘价, 最高价, 最低价, 开盘价]
    
#     # 资产1：AAPL  
#     [[1.02, 1.05, 0.98, 1.01]],
    
#     # 资产2：ADBE
#     [[0.98, 1.00, 0.95, 0.99]],
    
#     # 资产3：BABA
#     [[1.05, 1.08, 1.02, 1.06]],
    
#     # 资产4：SNE
#     [[0.99, 1.01, 0.97, 1.00]],
    
#     # 资产5：V
#     [[1.01, 1.03, 0.99, 1.02]]
# ]
len(states),states

(504,
 [array([[[[1.0, 1.0, 1.0, 1.0]],
  
          [[0.9173717838024521, 0.9380935935071664, 0.9101191504058022,
            0.9349853220514592]],
  
          [[0.6991743564837299, 0.7035454103933949, 0.6951918406993686,
            0.7014084507042253]],
  
          [[1.150210682154652, 1.173214895797745, 1.1376836351212847,
            1.1702539574080402]],
  
          [[0.7227970032108455, 0.7295754548697824, 0.7210132001427042,
            0.7295754548697824]],
  
          [[0.8304280953601642, 0.8442706998205589, 0.8295308895155089,
            0.8442706998205589]]]], dtype=object),
  array([[[[1.0, 1.0, 1.0, 1.0]],
  
          [[0.9174581246762218, 0.92756000690727, 0.90338456225177,
            0.9198756691417718]],
  
          [[0.6850898494414764, 0.7003399708596405, 0.6787761049052937,
            0.6988829528897523]],
  
          [[1.1766313631704817, 1.1826671221956495, 1.1400751622822,
            1.1530577382985991]],
  
          [[0.7224402425972173, 0.734213342

In [67]:
len(price_history),price_history

(504,
 [array([[1],
         [1],
         [1],
         [1],
         [1],
         [1]]),
  array([[1.        ],
         [1.00009412],
         [0.97985552],
         [1.0229703 ],
         [0.99950642],
         [0.99367186]]),
  array([[1.        ],
         [1.01402221],
         [1.00822345],
         [0.98848238],
         [1.06320988],
         [1.01335819]]),
  array([[1.        ],
         [1.03842227],
         [1.02545352],
         [1.02839518],
         [1.0013934 ],
         [1.01333538]]),
  array([[1.        ],
         [1.00107248],
         [0.98518925],
         [0.98086261],
         [0.98979592],
         [0.98517622]]),
  array([[1.        ],
         [0.97535934],
         [0.98997773],
         [0.98641041],
         [1.00093721],
         [0.99800399]]),
  array([[1.        ],
         [1.00887872],
         [0.99254781],
         [0.9916355 ],
         [0.99157303],
         [1.00292308]]),
  array([[1.        ],
         [0.99618944],
         [0.99150021],